# Experiment 2 - Matched Parameters

By default the two libraries run at different ring dimensions (8192 for TenSEAL, 16384 for OpenFHE) and different scaling factors, so a direct comparison measures the settings as well as the code.

This experiment pins OpenFHE to TenSEAL's ring dimension and scaling factor, and sweeps OpenFHE's four scaling techniques.

Corresponds to Experiment 2 in the report, *The Libraries, or the Parameters?*

## 1. Install and load the project


In [1]:
import subprocess
import sys

# TenSEAL publishes real platform-tagged wheels (win_amd64 / manylinux / macosx,
# CPython 3.8-3.12), so it installs anywhere. Pinned to the version the report used.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tenseal==0.3.15", "numpy", "matplotlib"], check=True)

# OpenFHE is the opposite. Every OpenFHE distribution on PyPI is tagged
# `py3-none-any`, but the payload is Linux ELF binaries built against one
# specific CPython. The 1.4.1 line, which the report used, is:
#
#     openfhe==1.4.1.0.20.4   Ubuntu 20.04   cpython-38    requires_python >=3.8
#     openfhe==1.4.1.0.22.4   Ubuntu 22.04   cpython-310   requires_python >=3.10
#     openfhe==1.4.1.0.24.4   Ubuntu 24.04   cpython-312   requires_python >=3.12
#
# Those bounds are `>=`, so an unpinned install takes the newest wheel whose
# bound this interpreter satisfies -- which need not be built for it. On 3.10
# and 3.12 that happens to land on a valid build; on 3.9, 3.11 and 3.13 it does
# not (3.11 resolves to the cpython-310 wheel openfhe==1.5.1.0.22.4), installs
# with no warning, and fails at *import*. Pinning also keeps the OpenFHE version
# identical to the report's.
OPENFHE_WHEELS = {(3, 8): "1.4.1.0.20.4", (3, 10): "1.4.1.0.22.4", (3, 12): "1.4.1.0.24.4"}
_py = sys.version_info[:2]
_wheel = OPENFHE_WHEELS.get(_py)

if sys.platform != "linux":
    print(f"Skipping OpenFHE: no wheel has ever been published for {sys.platform!r}. "
          "The plaintext and TenSEAL columns still run.")
elif _wheel is None:
    _have = ", ".join(f"{a}.{b}" for a, b in sorted(OPENFHE_WHEELS))
    print(f"Skipping OpenFHE: no build exists for CPython {_py[0]}.{_py[1]} "
          f"(builds exist for {_have}). The plaintext and TenSEAL columns still run.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"openfhe=={_wheel}"], check=True)
    print(f"Installed openfhe=={_wheel} for CPython {_py[0]}.{_py[1]}.")

Skipping OpenFHE: no wheel has ever been published for 'win32'. The plaintext and TenSEAL columns still run.


In [2]:
import os
import subprocess
import sys

REPO = "https://github.com/To2004/confidential-computing-project.git"

# Works both on a fresh Colab runtime and inside a local checkout.
if not os.path.exists("src/benchmark_harness.py"):
    if not os.path.exists("confidential-computing-project"):
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("confidential-computing-project")

# Absolute, so imports survive a later change of directory, and guarded so
# re-running this cell does not stack duplicate entries.
SRC = os.path.abspath("src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Write this notebook's output to its own directory, so running it does not
# overwrite the results and figures the report was built from.
import benchmark_harness as harness
import project_paths

os.makedirs("notebook_output", exist_ok=True)
project_paths.RESULTS_DIR = "notebook_output"
project_paths.FIGURES_DIR = "notebook_output"

print("working directory:", os.getcwd())
print("report uses", harness.DEFAULT_REPEATS, "repetitions per measurement")

working directory: c:\Users\user\Desktop\coursework\confidential-computing-project
report uses 1000 repetitions per measurement


## 2. What this experiment changes

Experiment 1 found TenSEAL faster on every arithmetic operation. That comparison was
confounded. OpenFHE derived ring dimension **16384** and a **2^50** scale from the depth and
security level it was asked for, against TenSEAL's **8192** and **2^40** - roughly twice the
polynomial work per operation, while carrying ten more bits of precision per value. OpenFHE was
not slower at the same job; it was doing a larger one.

This experiment re-runs OpenFHE pinned to TenSEAL's parameters, and additionally sweeps
OpenFHE's **scaling technique** - an internal control over how it manages the scaling factor
between multiplications, which TenSEAL does not expose at all.

In [ ]:
import notebook_tools as nt

nt.show_source(
    "matched_comparison.py", "openfhe_arm",
    note="The two lines that carry this experiment are `params.SetRingDim(ring_dim)` and "
         "`params.SetSecurityLevel(SecurityLevel.HEStd_NotSet)`. Everything else is the same "
         "operation dict Experiment 1 uses.")

### The trade this experiment makes, stated plainly

Asked for a ring dimension of 8192 at these moduli, OpenFHE **raises an error** rather than
accepting them: 8192 does not comply with the 128-bit standard here. The matched arm runs only
because the security level was explicitly relaxed to `HEStd_NotSet`, the setting OpenFHE
documents for prototyping.

So this experiment reports what the libraries do at **equal** parameters, not at equally
**secure** ones. That is a deliberate trade to isolate one variable, and it is disclosed as
such in the report. `probe_library_chosen_ring` records what OpenFHE picks when left alone.

In [ ]:
nt.show_source(
    "matched_comparison.py", "probe_library_chosen_ring",
    note="Run without a pinned ring dimension, so the returned value is what the library "
         "considers the smallest 128-bit-secure ring for these moduli.")

## 3. Check which libraries loaded

OpenFHE only publishes Linux binaries, and each build targets one specific
CPython (3.8, 3.10 or 3.12). On any other platform or Python version the
install cell above skips it, so it will not be listed as available here.

This experiment needs OpenFHE and will not run without it.

In [3]:
import importlib

for name in ("tenseal", "openfhe"):
    try:
        importlib.import_module(name)
        print(f"{name}: available")
    except Exception as exc:
        print(f"{name}: not available -- {exc}")

# This experiment measures OpenFHE itself, so stop here with a clear message
# rather than letting the benchmark below fail inside a subprocess.
try:
    import openfhe
except ImportError:
    raise RuntimeError(
        "This notebook measures OpenFHE itself, and OpenFHE did not import. "
        "OpenFHE publishes Linux-only wheels, each built against one specific "
        "CPython (3.8, 3.10 or 3.12 -- see the install cell above); it cannot "
        "run on Windows or macOS at all. Use WSL2 or a Linux runtime whose "
        "Python matches one of those builds, or the he38 environment described "
        "in the project README."
    ) from None

tenseal: available
openfhe: not available -- No module named 'openfhe'


RuntimeError: This notebook measures OpenFHE itself, and OpenFHE did not import. OpenFHE publishes Linux-only wheels, each built against one specific CPython (3.8, 3.10 or 3.12 -- see the install cell above); it cannot run on Windows or macOS at all. Use WSL2 or a Linux runtime whose Python matches one of those builds, or the he38 environment described in the project README.

## 4. Run the experiment

`REPEATS` is the number of timed repetitions per measurement. The report uses 1000 on a reserved compute node; this notebook uses fewer so it finishes in a few minutes. The numbers it prints will therefore be noisier than the ones in the report.

In [ ]:
REPEATS = 30

!"{sys.executable}" src/matched_comparison.py --repeats {REPEATS} --warmup 5 --output notebook_output/matched_comparison_results.json

## 5. Results


In [ ]:
import plot_results as pr
from IPython.display import Image, display

pr.apply_style()
pr.chart_matched(pr.load("matched_comparison_results.json"))
display(Image("notebook_output/chart_matched_comparison.png"))